In [3]:
from DocumentHandler import DocumentHandler
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from dotenv import load_dotenv 
import pandas as pd
import os

d:\machine learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
embedding = HuggingFaceEmbeddings(
    model_name = 'BAAI/bge-large-en-v1.5',
    cache_folder='.model',
    model_kwargs={"device": "cpu", "local_files_only": False},
)

Loading weights: 100%|██████████| 391/391 [00:01<00:00, 291.04it/s]


In [5]:
df = pd.read_csv("../../datasets/WattBot2026/metadata.csv")

In [6]:
document_handler = DocumentHandler(metadata_csv="../../datasets/WattBot2026/metadata.csv")

chunks:list[Document] = []
for doc_id in df['id']:
    chunks.extend(document_handler.get_chunks_from_document(doc_id))


rag.document.loading_started | document_id=amazon2023
rag.document.processing_completed | document_id=amazon2023 | chunks=334
rag.document.loading_started | document_id=chen2024
rag.document.processing_completed | document_id=chen2024 | chunks=35
rag.document.loading_started | document_id=chung2025
rag.document.processing_completed | document_id=chung2025 | chunks=34
rag.document.loading_started | document_id=cottier2024
rag.document.processing_completed | document_id=cottier2024 | chunks=37
rag.document.loading_started | document_id=dodge2022
rag.document.processing_completed | document_id=dodge2022 | chunks=31
rag.document.loading_started | document_id=ebert2024
rag.document.processing_completed | document_id=ebert2024 | chunks=41
rag.document.loading_started | document_id=erben2023
rag.document.processing_completed | document_id=erben2023 | chunks=32
rag.document.loading_started | document_id=fernandez2025
rag.document.processing_completed | document_id=fernandez2025 | chunks=28
rag

In [7]:
len(chunks)

4315

In [8]:
print(chunks[350].model_dump_json())

{"id":null,"metadata":{"id":"chen2024","type":"paper","title":"Efficient Heterogeneous Large Language Model Decoding with Model-Attention Disaggregation","year":"2024","citation":"Shaoyuan Chen, Wencong Xiao, Yutong Lin, Mingxing Zhang, Yingdi Shan, Jinlei Jiang, Kang Chen, Yongwei Wu. (2024). Efficient Heterogeneous Large Language Model Decoding with Model-Attention Disaggregation. arXiv. https://arxiv.org/pdf/2405.01814","url":"https://arxiv.org/pdf/2405.01814v2","peer_reviewed":"preprint","venue_evidence":"on arXiv with no journal_ref and no DOI","venue":"","section_headings":"**Efficient Heterogeneous Large Language Model Decoding with Model-Attention Disaggregation**, **4.1 Fully Host-Bypassed Network Stack**"},"page_content":"type-paper\ntitle-Efficient Heterogeneous Large Language Model Decoding with Model-Attention Disaggregation\nyear-2024\ncitation-Shaoyuan Chen, Wencong Xiao, Yutong Lin, Mingxing Zhang, Yingdi Shan, Jinlei Jiang, Kang Chen, Yongwei Wu. (2024). Efficient Hete

In [11]:
from hashlib import sha256
from typing import Iterable

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

from Elasticsearch.ElasticSearchClient import es


# ---------------------------------------------------------
# Embedding model
# ---------------------------------------------------------

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    cache_folder=".model",
    model_kwargs={
        "device": "cpu",
        "local_files_only": False,
    },
)


# ---------------------------------------------------------
# LangChain Document -> Elasticsearch document
# ---------------------------------------------------------

def store_document(
    document: Document,
    index_name: str,
) -> dict:
    """
    Convert one LangChain Document into an Elasticsearch document,
    generate its embedding, and store it.

    Elasticsearch mapping:
        page_content       -> text
        metadata           -> object
        content_embedding  -> dense_vector(1024)
    """

    if not isinstance(document, Document):
        raise TypeError("document must be a LangChain Document")

    if not document.page_content or not document.page_content.strip():
        raise ValueError("Document page_content is empty")

    # Generate 1024-dimensional embedding
    content_embedding = embedding.embed_query(
        document.page_content
    )

    if len(content_embedding) != 1024:
        raise ValueError(
            f"Expected 1024-dimensional embedding, "
            f"got {len(content_embedding)}"
        )

    # -----------------------------------------------------
    # Preserve LangChain metadata exactly
    # -----------------------------------------------------

    metadata = dict(document.metadata)

    # -----------------------------------------------------
    # Create a deterministic unique Elasticsearch _id.
    #
    # metadata["id"] is the paper/document ID and can occur
    # in multiple chunks, so it must NOT be used directly
    # as the Elasticsearch _id.
    # -----------------------------------------------------

    elastic_id = str(metadata.get("id", "document"))

    # -----------------------------------------------------
    # Final Elasticsearch document
    # -----------------------------------------------------

    es_document = {
        "page_content": document.page_content,
        "metadata": metadata,
        "content_embedding": content_embedding,
    }

    return es.add_document(
        index_name=index_name,
        document=es_document,
        document_id=elastic_id,
    )


# ---------------------------------------------------------
# Store multiple LangChain Documents efficiently
# ---------------------------------------------------------

def store_documents(
    documents: Iterable[Document],
    index_name: str,
) -> None:
    """
    Generate embeddings for multiple LangChain Documents and
    store them in Elasticsearch.
    """

    documents = list(documents)

    if not documents:
        return

    for document in documents:
        if not isinstance(document, Document):
            raise TypeError("All items must be LangChain Document objects")

        if not document.page_content or not document.page_content.strip():
            raise ValueError("Document page_content is empty")

    # Batch embedding generation
    texts = [document.page_content for document in documents]

    embeddings = embedding.embed_documents(texts)

    if any(len(vector) != 1024 for vector in embeddings):
        raise ValueError("BGE-large-en-v1.5 must produce 1024-dimensional vectors")

    # Prepare Elasticsearch bulk actions
    operations = []

    for document, vector in zip(documents, embeddings):
        metadata = dict(document.metadata)

        elastic_id = str(metadata.get("id", "document"))

        operations.append(
            {
                "index": {
                    "_index": index_name,
                    "_id": elastic_id,
                }
            }
        )

        operations.append(
            {
                "page_content": document.page_content,
                "metadata": metadata,
                "content_embedding": vector,
            }
        )

    es.get_client().bulk(operations=operations, refresh=True)

ModuleNotFoundError: No module named 'WattBot_2026'